# Wakefield - Comparing to Bmad

A resistive-wall wakefield built with [openPMD-beamphysics](https://github.com/ChristopherMayes/openPMD-beamphysics) is attached to a pipe in Bmad, tracked, and then compared with IMPACT-Z.

`ImpactZ.from_tao` reads the short-range wake back out of Tao and converts it into the read-in wake table that the IMPACT-Z `RfcavityStructureWakefield` element (type `-41`) expects, so no manual translation is needed.


In [ ]:
import matplotlib.pyplot as plt
from pytao import Tao
from pytao.model import BeamInit
from scipy.constants import c

from beamphysics.wakefields import ResistiveWallPseudomode

import impact.z as IZ

In [ ]:
wf = ResistiveWallPseudomode.from_material("copper-slac-pub-10707", radius=0.0045)
wf.plot()

In [ ]:
LAT_STR = f"""
beginning[beta_a] = 1
beginning[beta_b] = 1
parameter[geometry] = open
parameter[e_tot] = 100e6
parameter[particle] = electron


p1: pipe, L = 2, sr_wake = {wf.to_bmad()}

lat: line = (p1)

use, lat
"""

print(LAT_STR)

In [ ]:
tao = Tao.from_lattice_contents(LAT_STR, plot="mpl")

In [ ]:
bi = BeamInit()
bi.bunch_charge = 1e-9
bi.n_particle = 100000
bi.a_norm_emit = 1e-9
bi.b_norm_emit = 1e-9
bi.sig_z = 10e-6
bi.sig_pz = 1e-6
bi.set(tao)

In [ ]:
tao.track_beam()
P1 = tao.particles("END")
P1.plot("t", "energy")
P1["mean_energy"]

## Impact-Z

The conversion picks the wake up automatically. Each Bmad pseudo-mode becomes a term of an openPMD-beamphysics `PseudomodeWakefield`, which is then sampled onto the four-column table that the `-41` element reads, and the element is bracketed by a pair of `-41` switches that turn the wake on and off.

Note that the beam current must be nonzero for IMPACT-Z to evaluate any collective effect, so the conversion sets it from the Bmad bunch charge whenever a wake is present. The space-charge solve stays off.


In [ ]:
I = IZ.ImpactZ.from_tao(tao)
I.input.lattice

In [ ]:
I.run()
P2 = I.output.particles["final_particles"]
P2.plot("t", "energy")
P2["mean_energy"]

## Compare


In [ ]:
P0 = tao.particles("BEGINNING")

for label, P in (("Bmad", P1), ("Impact-Z", P2)):
    print(
        f"{label:10s} mean energy loss: {(P0['mean_energy'] - P['mean_energy']) * 1e-3:8.3f} keV"
    )

In [ ]:
fig, ax = plt.subplots()
for label, P in (("Bmad", P1), ("Impact-Z", P2)):
    # Equal-population slices, so the sparse tails of the bunch are merged into
    # wide slices instead of leaving bins that hold few particles or none.
    ss = P.slice_statistics("mean_energy", n_slice=60, slice_key="t")
    z = -(ss["mean_t"] - P["mean_t"]) * c
    ax.plot(z * 1e6, (ss["mean_energy"] - P0["mean_energy"]) * 1e-3, label=label)

ax.set_xlabel(r"$z$ (µm)  (head to the right)")
ax.set_ylabel(r"$\Delta E$ (keV)")
ax.set_title("Wake-induced energy change")
ax.legend()